<center><img src="car.jpg" width=500></center>


Insurance companies invest a lot of time and money into optimizing their pricing and accurately estimating the likelihood that customers will make a claim. In many countries insurance it is a legal requirement to have car insurance in order to drive a vehicle on public roads, so the market is very large!

(`Source: https://www.accenture.com/_acnmedia/pdf-84/accenture-machine-leaning-insurance.pdf`) 

Knowing all of this, On the Road car insurance have requested your services in building a model to predict whether a customer will make a claim on their insurance during the policy period. As they have very little expertise and infrastructure for deploying and monitoring machine learning models, they've asked you to identify the single feature that results in the best performing model, as measured by accuracy, so they can start with a simple model in production.

They have supplied you with their customer data as a csv file called `car_insurance.csv`, along with a table detailing the column names and descriptions below.



## The dataset

| Column | Description |
|--------|-------------|
| `id` | Unique client identifier |
| `age` | Client's age: <br> <ul><li>`0`: 16-25</li><li>`1`: 26-39</li><li>`2`: 40-64</li><li>`3`: 65+</li></ul> |
| `gender` | Client's gender: <br> <ul><li>`0`: Female</li><li>`1`: Male</li></ul> |
| `driving_experience` | Years the client has been driving: <br> <ul><li>`0`: 0-9</li><li>`1`: 10-19</li><li>`2`: 20-29</li><li>`3`: 30+</li></ul> |
| `education` | Client's level of education: <br> <ul><li>`0`: No education</li><li>`1`: High school</li><li>`2`: University</li></ul> |
| `income` | Client's income level: <br> <ul><li>`0`: Poverty</li><li>`1`: Working class</li><li>`2`: Middle class</li><li>`3`: Upper class</li></ul> |
| `credit_score` | Client's credit score (between zero and one) |
| `vehicle_ownership` | Client's vehicle ownership status: <br><ul><li>`0`: Does not own their vehilce (paying off finance)</li><li>`1`: Owns their vehicle</li></ul> |
| `vehcile_year` | Year of vehicle registration: <br><ul><li>`0`: Before 2015</li><li>`1`: 2015 or later</li></ul> |
| `married` | Client's marital status: <br><ul><li>`0`: Not married</li><li>`1`: Married</li></ul> |
| `children` | Client's number of children |
| `postal_code` | Client's postal code | 
| `annual_mileage` | Number of miles driven by the client each year |
| `vehicle_type` | Type of car: <br> <ul><li>`0`: Sedan</li><li>`1`: Sports car</li></ul> |
| `speeding_violations` | Total number of speeding violations received by the client | 
| `duis` | Number of times the client has been caught driving under the influence of alcohol |
| `past_accidents` | Total number of previous accidents the client has been involved in |
| `outcome` | Whether the client made a claim on their car insurance (response variable): <br><ul><li>`0`: No claim</li><li>`1`: Made a claim</li></ul> |



    Identify the single feature of the data that is the best predictor of whether a customer will put in a claim (the "outcome" column), excluding the "id" column.
    Store as a DataFrame called best_feature_df, containing columns named "best_feature" and "best_accuracy" with the name of the feature with the highest accuracy, and the respective accuracy score. 

In [18]:
# Import required modules
import pandas as pd
import numpy as np
from statsmodels.formula.api import logit

# Read and inspect the data
car_insurance = pd.read_csv('car_insurance.csv')
print(car_insurance.isna().sum())
print(car_insurance.info())

id                       0
age                      0
gender                   0
driving_experience       0
education                0
income                   0
credit_score           982
vehicle_ownership        0
vehicle_year             0
married                  0
children                 0
postal_code              0
annual_mileage         957
vehicle_type             0
speeding_violations      0
duis                     0
past_accidents           0
outcome                  0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   10000 non-null  int64  
 1   age                  10000 non-null  int64  
 2   gender               10000 non-null  int64  
 3   driving_experience   10000 non-null  object 
 4   education            10000 non-null  object 
 5   income               10000 non-null  o

In [19]:
# impute the missing values using the mean
car_insurance["credit_score"].fillna(car_insurance["credit_score"].mean(), inplace=True)
car_insurance["annual_mileage"].fillna(car_insurance["annual_mileage"].mean(), inplace=True)
car_insurance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   10000 non-null  int64  
 1   age                  10000 non-null  int64  
 2   gender               10000 non-null  int64  
 3   driving_experience   10000 non-null  object 
 4   education            10000 non-null  object 
 5   income               10000 non-null  object 
 6   credit_score         10000 non-null  float64
 7   vehicle_ownership    10000 non-null  float64
 8   vehicle_year         10000 non-null  object 
 9   married              10000 non-null  float64
 10  children             10000 non-null  float64
 11  postal_code          10000 non-null  int64  
 12  annual_mileage       10000 non-null  float64
 13  vehicle_type         10000 non-null  object 
 14  speeding_violations  10000 non-null  int64  
 15  duis                 10000 non-null  

In [20]:
# Now lets make sure that the expected number of unique values match for each column
for col in car_insurance:
    print('{} has {} unique values'.format(col, car_insurance[col].nunique()))
    
    expected_nunique = {
    "id": "Unique for each client (number of rows)",
    "age": 4,
    "gender": 2,
    "driving_experience": 4,
    "education": 3,
    "income": 4,
    "credit_score": "Many (continuous values between 0 and 1)",
    "vehicle_ownership": 2,
    "vehcile_year": 2,
    "married": 2,
    "children": "Depends on dataset (typically a small set of counts)",
    "postal_code": "Depends on dataset (could be many unique codes)",
    "annual_mileage": "Depends on data type (likely many if continuous)",
    "vehicle_type": 2,
    "speeding_violations": "Likely a few distinct counts (count data)",
    "duis": "Likely a few distinct counts (count data)",
    "past_accidents": "Likely a few distinct counts (count data)",
    "outcome": 2
}


id has 10000 unique values
age has 4 unique values
gender has 2 unique values
driving_experience has 4 unique values
education has 3 unique values
income has 4 unique values
credit_score has 9019 unique values
vehicle_ownership has 2 unique values
vehicle_year has 2 unique values
married has 2 unique values
children has 2 unique values
postal_code has 4 unique values
annual_mileage has 22 unique values
vehicle_type has 2 unique values
speeding_violations has 21 unique values
duis has 7 unique values
past_accidents has 15 unique values
outcome has 2 unique values


In [21]:
# Create an empty list to contain the logit models we create from our features (columns)
models = []
features = car_insurance.columns.drop(['id', 'outcome'])

# Create a for loop that fits each feature to a model and stores it in the models list
for col in features:
    formula = f'outcome ~ {col}'
    model = logit(formula, data=car_insurance).fit()
    models.append(model)

Optimization terminated successfully.
         Current function value: 0.511794
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.615951
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.467092
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.603742
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.531499
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.572557
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.552412
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.572668
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.586659
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.595431
  

In [22]:
# Create a list to store the accuracies of the models
accuracies = []

# Using a for loop and thr prediction tables of each model calculate the accuracies of each model and store it in the accuracies list 
for i in range(len(models)):
    conf_matrix = models[i].pred_table()
    TN = conf_matrix[0, 0]
    FP = conf_matrix[0, 1]
    FN = conf_matrix[1, 0]
    TP = conf_matrix[1, 1]
    accuracy = (TP + TN) / conf_matrix.sum()
    accuracies.append(accuracy)

In [23]:
accuracies

[0.7747,
 0.6867,
 0.7771,
 0.6867,
 0.7425,
 0.7054,
 0.7351,
 0.6867,
 0.6867,
 0.6867,
 0.6867,
 0.6904,
 0.6867,
 0.6867,
 0.6867,
 0.6867]

In [24]:
# Locate the best features name and accuracy value and store them in a dataframe
best_index = accuracies.index(max(accuracies))
best_feature = features[best_index]
best_accuracy = accuracies[best_index]

best_feature_df = pd.DataFrame({
    'best_feature': [best_feature],
    'best_accuracy': [best_accuracy]
}, index=[0])

print("Best Model:")
print(best_feature_df)

Best Model:
         best_feature  best_accuracy
0  driving_experience         0.7771
